<a href="https://colab.research.google.com/github/khildasa/PengolahanCitra_2025/blob/main/TugasBesar_KhildaSalsabilaA_PengolahanCitra.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **TUGAS BESAR - PENGOLAHAN CITRA - AI VIRTUAL MOUSE**
---
Khilda Salsabila Azka || 4.33.23.0.15 || TI-2A

## 1. Install Required Libraries

In [ ]:
# Install required packages
!pip install opencv-python
!pip install mediapipe
!pip install autopy
!pip install numpy


[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: C:\Users\Khilda\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: C:\Users\Khilda\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: C:\Users\Khilda\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: C:\Users\Khilda\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 2. Camera Test

In [ ]:
import cv2
import numpy as np
import time

# Test kamera terlebih dahulu
def test_camera():
    print("Testing kamera...")
    cap = cv2.VideoCapture(0)

    if not cap.isOpened():
        print("Error: Tidak dapat membuka kamera!")
        print("Coba ganti dengan cv2.VideoCapture(1) atau cv2.VideoCapture(2)")
        return False

    print("Kamera berhasil dibuka!")

    # Test capture frame
    ret, frame = cap.read()
    if ret:
        print(f"Resolusi kamera: {frame.shape[1]}x{frame.shape[0]}")
        cv2.imshow("Test Kamera - Tekan 'q' untuk keluar", frame)
        cv2.waitKey(2000)  # Tampilkan selama 2 detik
        cv2.destroyAllWindows()

    cap.release()
    return True

# Jalankan test
test_camera()

Testing kamera...
Kamera berhasil dibuka!
Resolusi kamera: 640x480


True

## 3. Hand Tracking Module

In [ ]:
# HandTrackingModule.py
import cv2
import mediapipe as mp
import time
import math
import numpy as np

class handDetector():
    def __init__(self, mode=False, maxHands=2, detectionCon=0.5, trackCon=0.5):
        self.mode = mode
        self.maxHands = maxHands
        self.detectionCon = detectionCon
        self.trackCon = trackCon

        self.mpHands = mp.solutions.hands
        self.hands = self.mpHands.Hands(
            static_image_mode=self.mode,
            max_num_hands=self.maxHands,
            min_detection_confidence=self.detectionCon,
            min_tracking_confidence=self.trackCon
        )
        self.mpDraw = mp.solutions.drawing_utils
        self.tipIds = [4, 8, 12, 16, 20]

    def findHands(self, img, draw=True):
        imgRGB = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        self.results = self.hands.process(imgRGB)

        if self.results.multi_hand_landmarks:
            for handLms in self.results.multi_hand_landmarks:
                if draw:
                    self.mpDraw.draw_landmarks(img, handLms,
                                               self.mpHands.HAND_CONNECTIONS)
        return img

    def findPosition(self, img, handNo=0, draw=True):
        xList = []
        yList = []
        bbox = []
        self.lmList = []

        if self.results.multi_hand_landmarks:
            myHand = self.results.multi_hand_landmarks[handNo]
            for id, lm in enumerate(myHand.landmark):
                h, w, c = img.shape
                cx, cy = int(lm.x * w), int(lm.y * h)
                xList.append(cx)
                yList.append(cy)
                self.lmList.append([id, cx, cy])
                if draw:
                    cv2.circle(img, (cx, cy), 5, (255, 0, 255), cv2.FILLED)

            xmin, xmax = min(xList), max(xList)
            ymin, ymax = min(yList), max(yList)
            bbox = xmin, ymin, xmax, ymax

            if draw:
                cv2.rectangle(img, (xmin - 20, ymin - 20), (xmax + 20, ymax + 20),
                              (0, 255, 0), 2)

        return self.lmList, bbox

    def fingersUp(self):
        fingers = []

        if len(self.lmList) == 0:
            return []

        # Thumb
        if self.lmList[self.tipIds[0]][1] > self.lmList[self.tipIds[0] - 1][1]:
            fingers.append(1)
        else:
            fingers.append(0)

        # Fingers
        for id in range(1, 5):
            if self.lmList[self.tipIds[id]][2] < self.lmList[self.tipIds[id] - 2][2]:
                fingers.append(1)
            else:
                fingers.append(0)

        return fingers

    def findDistance(self, p1, p2, img, draw=True, r=15, t=3):
        if len(self.lmList) == 0:
            return 0, img, []

        x1, y1 = self.lmList[p1][1:]
        x2, y2 = self.lmList[p2][1:]
        cx, cy = (x1 + x2) // 2, (y1 + y2) // 2

        if draw:
            cv2.line(img, (x1, y1), (x2, y2), (255, 0, 255), t)
            cv2.circle(img, (x1, y1), r, (255, 0, 255), cv2.FILLED)
            cv2.circle(img, (x2, y2), r, (255, 0, 255), cv2.FILLED)
            cv2.circle(img, (cx, cy), r, (0, 0, 255), cv2.FILLED)

        length = math.hypot(x2 - x1, y2 - y1)
        return length, img, [x1, y1, x2, y2, cx, cy]

print("HandTrackingModule berhasil dibuat!")

HandTrackingModule berhasil dibuat!


## 4.  Test Hand Detection

In [ ]:
# Test deteksi tangan
def test_hand_detection():
    print("Testing hand detection...")
    print("Tunjukkan tangan Anda ke kamera")
    print("Tekan 'q' untuk keluar")

    cap = cv2.VideoCapture(0)
    detector = handDetector(maxHands=1)

    while True:
        success, img = cap.read()
        if not success:
            print("Gagal membaca frame dari kamera")
            break

        img = detector.findHands(img)
        lmList, bbox = detector.findPosition(img)

        if len(lmList) != 0:
            # Tampilkan informasi jari yang terangkat
            fingers = detector.fingersUp()
            cv2.putText(img, f"Fingers: {fingers}", (10, 70),
                       cv2.FONT_HERSHEY_PLAIN, 2, (255, 0, 0), 3)

        cv2.imshow("Hand Detection Test", img)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    print("Test selesai!")

test_hand_detection()

Testing hand detection...
Tunjukkan tangan Anda ke kamera
Tekan 'q' untuk keluar
Test selesai!


## 5.  AI Virtual Mouse

In [ ]:
import autopy

def run_virtual_mouse():
    print("=== AI VIRTUAL MOUSE (IMPROVED) ===")
    print("Instruksi penggunaan:")
    print("1. Angkat HANYA jari telunjuk untuk menggerakkan mouse")
    print("2. Angkat jari telunjuk DAN jari tengah untuk mode klik")
    print("3. Dekatkan kedua jari untuk melakukan klik (sekali saja)")
    print("4. Tekan 'q' untuk keluar")
    time.sleep(3)

    ##########################
    wCam, hCam = 640, 480
    frameR = 100  # Frame Reduction
    smoothening = 7
    clickCooldown = 20  # Cooldown untuk mencegah spam klik
    #########################

    pTime = 0
    plocX, plocY = 0, 0
    clocX, clocY = 0, 0
    clickCounter = 0  # Counter untuk cooldown klik
    hasClicked = False  # Flag untuk mencegah spam klik

    cap = cv2.VideoCapture(0)  # Ubah ke 1 jika kamera tidak terdeteksi
    cap.set(3, wCam)
    cap.set(4, hCam)

    detector = handDetector(maxHands=1)
    wScr, hScr = autopy.screen.size()
    print(f"Resolusi layar: {wScr} x {hScr}")

    try:
        while True:
            # 1. Deteksi tangan
            success, img = cap.read()
            if not success:
                print("Gagal membaca frame")
                break

            # FIX 1: FLIP KAMERA HORIZONTAL untuk mengatasi mirror effect
            img = cv2.flip(img, 1)

            img = detector.findHands(img)
            lmList, bbox = detector.findPosition(img)

            # Kurangi click counter
            if clickCounter > 0:
                clickCounter -= 1

            # 2. Dapatkan posisi jari telunjuk dan tengah
            if len(lmList) != 0:
                x1, y1 = lmList[8][1:]  # Jari telunjuk
                x2, y2 = lmList[12][1:]  # Jari tengah

                # 3. Cek jari mana yang terangkat
                fingers = detector.fingersUp()

                # Gambar area kontrol
                cv2.rectangle(img, (frameR, frameR), (wCam - frameR, hCam - frameR),
                            (255, 0, 255), 2)

                # 4. Mode Moving - Hanya jari telunjuk
                if len(fingers) >= 3 and fingers[1] == 1 and fingers[2] == 0:
                    # Reset flag klik karena keluar dari mode klik
                    hasClicked = False

                    # 5. Konversi koordinat (sudah tidak perlu flip karena img sudah di-flip)
                    x3 = np.interp(x1, (frameR, wCam - frameR), (0, wScr))
                    y3 = np.interp(y1, (frameR, hCam - frameR), (0, hScr))

                    # 6. Smoothing
                    clocX = plocX + (x3 - plocX) / smoothening
                    clocY = plocY + (y3 - plocY) / smoothening

                    # 7. Gerakkan mouse
                    try:
                        autopy.mouse.move(clocX, clocY)  # Tidak perlu flip lagi
                    except:
                        pass  # Ignore errors saat move mouse

                    cv2.circle(img, (x1, y1), 15, (255, 0, 255), cv2.FILLED)
                    plocX, plocY = clocX, clocY

                    # Tampilkan status
                    cv2.putText(img, "MOVING MODE", (10, 110),
                               cv2.FONT_HERSHEY_PLAIN, 2, (0, 255, 0), 3)

                # 8. Mode Clicking - Jari telunjuk dan tengah
                elif len(fingers) >= 3 and fingers[1] == 1 and fingers[2] == 1:
                    # 9. Hitung jarak antar jari
                    length, img, lineInfo = detector.findDistance(8, 12, img)

                    # 10. Klik
                    if length < 40:
                        cv2.circle(img, (lineInfo[4], lineInfo[5]),
                                 15, (0, 255, 0), cv2.FILLED)

                        # Klik hanya jika belum pernah klik dan cooldown habis
                        if not hasClicked and clickCounter == 0:
                            try:
                                autopy.mouse.click()
                                hasClicked = True  # Set flag sudah klik
                                clickCounter = clickCooldown  # Set cooldown
                                print("CLICK!")  # Debug info
                            except:
                                pass  # Ignore click errors

                        # Tampilkan status klik
                        cv2.putText(img, "CLICKING!", (10, 150),
                                   cv2.FONT_HERSHEY_PLAIN, 2, (0, 255, 0), 3)
                    else:
                        # Reset flag jika jari menjauh
                        if hasClicked and length > 50:
                            hasClicked = False

                    # Tampilkan status
                    cv2.putText(img, f"CLICK MODE - Distance: {int(length)}",
                               (10, 110), cv2.FONT_HERSHEY_PLAIN, 2, (0, 0, 255), 3)
                else:
                    # Reset flag jika tidak ada jari yang terangkat
                    hasClicked = False

            # 11. Tampilkan FPS
            cTime = time.time()
            fps = 1 / (cTime - pTime) if (cTime - pTime) > 0 else 0
            pTime = cTime
            cv2.putText(img, f"FPS: {int(fps)}", (20, 50),
                       cv2.FONT_HERSHEY_PLAIN, 3, (255, 0, 0), 3)

            # Tampilkan status cooldown
            if clickCounter > 0:
                cv2.putText(img, f"Cooldown: {clickCounter}", (10, 190),
                           cv2.FONT_HERSHEY_PLAIN, 2, (255, 255, 0), 2)

            # 12. Tampilkan gambar
            cv2.imshow("AI Virtual Mouse (Fixed)", img)

            # Keluar dengan 'q'
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

    except KeyboardInterrupt:
        print("\nProgram dihentikan oleh user")
    except Exception as e:
        print(f"Error: {e}")
    finally:
        cap.release()
        cv2.destroyAllWindows()
        print("Program selesai!")

# Jalankan virtual mouse yang sudah diperbaiki
run_virtual_mouse()

=== AI VIRTUAL MOUSE (IMPROVED) ===
Instruksi penggunaan:
1. Angkat HANYA jari telunjuk untuk menggerakkan mouse
2. Angkat jari telunjuk DAN jari tengah untuk mode klik
3. Dekatkan kedua jari untuk melakukan klik (sekali saja)
4. Tekan 'q' untuk keluar
Resolusi layar: 1280.0 x 720.0
CLICK!
CLICK!
CLICK!
CLICK!
Program selesai!
